# Module 4 · NER clinique et OCR — Travaux pratiques
**Togo AI Lab · Formation IA · Version IT** · Durée : environ 1 h 30

**Ce que vous allez faire :** extraire automatiquement des informations de comptes rendus médicaux **fictifs** (problèmes, médicaments, doses, examens, dates), gérer la **négation**, mesurer la qualité, puis lire des documents **scannés** avec un OCR et observer comment ses erreurs se propagent jusqu'à l'extraction.

**À la fin, vous saurez :**
1. repérer des entités avec des règles et un dictionnaire, et voir leurs limites ;
2. détecter la négation, l'incertitude et l'antécédent familial ;
3. entraîner un petit étiqueteur de mots (format BIO) et l'évaluer **par entité** ;
4. lire une image avec Tesseract, mesurer le taux d'erreur de caractères (CER) et améliorer l'image ;
5. mesurer l'effet des erreurs d'OCR sur l'extraction.

**Comment utiliser ce notebook (Google Colab)**
- Exécutez les cellules avec **Maj + Entrée**, dans l'ordre. Aucun GPU n'est nécessaire.
- Les cellules **👉 À vous** contiennent des `____` à remplacer.
- **Toutes les données sont fictives.** Avec de vrais dossiers : autorisation, anonymisation et hébergement sécurisé sont obligatoires.

## 0 · Préparation

In [ ]:
import shutil, subprocess, sys
if shutil.which("tesseract") is None:
    !apt-get -qq update && apt-get -qq install -y tesseract-ocr tesseract-ocr-fra
try:
    import pytesseract
except ImportError:
    %pip install -q pytesseract

import re, random, unicodedata
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageOps
import pytesseract
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

random.seed(42); np.random.seed(42)
LANGUES = pytesseract.get_languages()
LANG = "fra" if "fra" in LANGUES else "eng"
print("Outils prêts ✔ — langue OCR :", LANG)

## 1 · Des comptes rendus fictifs
Un petit **générateur** fabrique des comptes rendus avec leurs annotations « vérité terrain » : pour chaque entité, sa position, son étiquette (`PROBLEME`, `MEDICAMENT`, `DOSE`, `EXAMEN`, `DATE`) et, pour les problèmes, un **statut** (`present`, `nie`, `incertain`, `famille`). **Exécutez la cellule sans la lire en détail.**

In [ ]:
# --- Générateur de comptes rendus fictifs (ne pas lire en détail) ---
PROB  = ["paludisme", "fièvre", "toux", "diarrhée", "vomissements", "hypertension", "diabète", "anémie", "œdème", "maux de tête", "déshydratation"]
MED   = ["paracétamol", "amoxicilline", "zinc", "amlodipine", "artésunate-amodiaquine", "fer", "metformine"]
EXA   = ["goutte épaisse", "glycémie", "tension artérielle", "test rapide", "radiographie"]
DOSES = ["500 mg", "5 mg", "20 mg", "3 fois par jour", "2 fois par jour", "pendant 3 jours", "pendant 5 jours", "pendant 10 jours", "un comprimé par jour"]

def date_alea(r):
    return f"{r.randint(1,28):02d}/{r.randint(1,12):02d}/2026"

def doc_alea(r):
    """Retourne (texte, entités) ; entité = (début, fin, étiquette, statut)."""
    parts = []
    parts.append(f"Patient {r.randint(1,999):03d}, {r.randint(2,80)} ans. Consulte le ")
    parts.append((date_alea(r), "DATE", None))
    p1, p2 = r.sample(PROB, 2)
    parts += [" pour ", (p1, "PROBLEME", "present"), " et ", (p2, "PROBLEME", "present"), f" depuis {r.randint(1,7)} jours. "]
    for _ in range(r.randint(1, 2)):
        k = r.choice(["nie", "nie", "incertain", "famille", "exam"])
        if k == "nie":
            parts += [r.choice(["Pas de ", "Sans ", "Ne présente pas de "]), (r.choice(PROB), "PROBLEME", "nie"), ". "]
        elif k == "incertain":
            parts += ["Suspicion de ", (r.choice(PROB), "PROBLEME", "incertain"), ". "]
        elif k == "famille":
            parts += ["Antécédent familial de ", (r.choice(PROB), "PROBLEME", "famille"), " (mère). "]
        else:
            parts += [(r.choice(EXA), "EXAMEN", None), r.choice([" à contrôler. ", " positive. ", " normale. "])]
    for _ in range(r.randint(1, 2)):
        parts += ["Traitement : ", (r.choice(MED), "MEDICAMENT", None), " ", (r.choice(DOSES), "DOSE", None)]
        if r.random() < 0.6:
            parts += [", ", (r.choice(DOSES), "DOSE", None)]
        parts += [". "]
    parts += ["Revoir dans ", (f"{r.randint(1,4)} semaines", "DATE", None), "."]
    texte, ents = "", []
    for p in parts:
        if isinstance(p, tuple):
            ents.append((len(texte), len(texte) + len(p[0]), p[1], p[2])); texte += p[0]
        else:
            texte += p
    return texte, ents

rg = random.Random(1)
train_docs = [doc_alea(rg) for _ in range(300)]
test_docs  = [doc_alea(rg) for _ in range(100)]
print(len(train_docs), "documents d'entraînement,", len(test_docs), "de test")
print()
t, e = test_docs[0]
print(t); print()
for d, f, l, s in e: print(f"  {l:11s} {t[d:f]!r:32} statut={s}")

**Un jeu de test écrit à la main.** Ces six comptes rendus utilisent **d'autres mots** que le générateur (maladies, médicaments, formulations inédits). Ils servent à vérifier si une méthode **généralise**.

In [ ]:
def doc_main(parts):
    texte, ents = "", []
    for p in parts:
        if isinstance(p, tuple):
            ents.append((len(texte), len(texte) + len(p[0]), p[1], p[2])); texte += p[0]
        else:
            texte += p
    return texte, ents

test_main = [
 doc_main(["Enfant 010, 6 ans. Fièvre typhoïde suspectée ? Non : ", ("typhoïde", "PROBLEME", "nie"), " écartée. ", ("céphalées", "PROBLEME", "present"), " persistantes. Prescription : ", ("ibuprofène", "MEDICAMENT", None), " ", ("200 mg", "DOSE", None), ", ", ("2 fois par jour", "DOSE", None), "."]),
 doc_main(["Patiente 011, 31 ans. ", ("Anémie", "PROBLEME", "present"), " modérée. ", ("Échographie", "EXAMEN", None), " prévue le ", ("15/05/2026", "DATE", None), ". Donner ", ("fer", "MEDICAMENT", None), " ", ("un comprimé par jour", "DOSE", None), "."]),
 doc_main(["Patient 012, 47 ans. Sans ", ("fièvre", "PROBLEME", "nie"), ". ", ("Asthme", "PROBLEME", "present"), " connu. ", ("Salbutamol", "MEDICAMENT", None), " ", ("2 fois par jour", "DOSE", None), "."]),
 doc_main(["Patiente 013, 25 ans, enceinte. ", ("Radiographie", "EXAMEN", None), " non nécessaire. Antécédent familial de ", ("drépanocytose", "PROBLEME", "famille"), ". Revoir dans ", ("2 semaines", "DATE", None), "."]),
 doc_main(["Patient 014, 62 ans. ", ("Paludisme", "PROBLEME", "present"), " grave. ", ("Artésunate", "MEDICAMENT", None), " ", ("pendant 3 jours", "DOSE", None), ". ", ("Glycémie", "EXAMEN", None), " à contrôler le ", ("04/06/2026", "DATE", None), "."]),
 doc_main(["Enfant 015, 3 ans. ", ("Toux", "PROBLEME", "present"), " sèche, pas de ", ("diarrhée", "PROBLEME", "nie"), ". Suspicion de ", ("pneumonie", "PROBLEME", "incertain"), ". ", ("Amoxicilline", "MEDICAMENT", None), " ", ("500 mg", "DOSE", None), " ", ("pendant 5 jours", "DOSE", None), "."]),
]
print(len(test_main), "comptes rendus écrits à la main")

## 2 · Première approche : des règles et un dictionnaire
Le plus simple : une **liste de termes** (dictionnaire) pour les problèmes, médicaments et examens, et des **expressions régulières** pour les doses et les dates.

👉 **À vous (1/4)** : complétez l'expression régulière des **doses** (`500 mg`, `3 fois par jour`, `pendant 5 jours`).

In [ ]:
# Dictionnaire connu du système (volontairement incomplet)
LEX = {
    "PROBLEME":   ["paludisme", "fièvre", "toux", "diarrhée", "vomissements", "hypertension", "diabète", "anémie", "œdème", "maux de tête"],
    "MEDICAMENT": ["paracétamol", "amoxicilline", "zinc", "amlodipine", "artésunate-amodiaquine", "fer"],
    "EXAMEN":     ["goutte épaisse", "glycémie", "tension artérielle", "test rapide", "radiographie"],
}
# 👉 Complétez : un nombre + mg, OU un nombre + "fois par jour", OU "pendant" + nombre + jours/semaines
RE_DOSE = r"\d+ (?:mg|fois par jour)|pendant \d+ (?:jours|semaines)|un comprimé par jour"
RE_DATE = r"\d{2}/\d{2}/\d{4}|dans \d+ semaines|\d+ semaines"

def extraire_regles(texte):
    ents = []
    for etiquette, termes in LEX.items():
        for t in sorted(termes, key=len, reverse=True):
            for m in re.finditer(r"\b" + re.escape(t) + r"\b", texte, flags=re.I):
                ents.append((m.start(), m.end(), etiquette))
    for etiquette, motif in (("DOSE", RE_DOSE), ("DATE", RE_DATE)):
        for m in re.finditer(motif, texte, flags=re.I):
            ents.append((m.start(), m.end(), etiquette))
    # retirer les chevauchements : on garde le plus long
    ents.sort(key=lambda e: (e[0], -(e[1] - e[0])))
    final, fin = [], -1
    for e in ents:
        if e[0] >= fin:
            final.append(e); fin = e[1]
    return final

t, e = test_docs[0]
for d, f, l in extraire_regles(t): print(f"{l:11s} {t[d:f]!r}")

### Mesurer par entité
Une entité n'est **correcte** que si ses **bornes exactes** et son **étiquette** sont justes (évaluation *stricte*). On compte :
- **précision** : parmi les entités trouvées, la part qui est juste (évite les fausses alertes) ;
- **rappel** : parmi les vraies entités, la part retrouvée (évite les oublis) ;
- **F1** : un compromis entre les deux.

In [ ]:
def score(docs, extracteur):
    """Score strict par entité sur une liste de (texte, entités)."""
    vp = fp = fn = 0
    for texte, ents in docs:
        vrai = {(d, f, l) for d, f, l, s in ents}
        pred = {(d, f, l) for d, f, l in extracteur(texte)}
        vp += len(vrai & pred); fp += len(pred - vrai); fn += len(vrai - pred)
    p = vp / (vp + fp) if vp + fp else 0
    r = vp / (vp + fn) if vp + fn else 0
    f1 = 2 * p * r / (p + r) if p + r else 0
    return {"précision": round(p, 2), "rappel": round(r, 2), "F1": round(f1, 2)}

print("Règles + dictionnaire, jeu de test généré :", score(test_docs, extraire_regles))
print("Règles + dictionnaire, jeu écrit à la main :", score(test_main, extraire_regles))

**À observer :** comparez les deux lignes. Le dictionnaire connaît les mots du générateur mais pas `typhoïde`, `ibuprofène` ou `drépanocytose` : le rappel baisse sur le jeu écrit à la main. Quelles autres erreurs voyez-vous ? Affichez les entités manquées :

In [ ]:
def voir_erreurs(docs, extracteur, n=6):
    k = 0
    for texte, ents in docs:
        vrai = {(d, f, l) for d, f, l, s in ents}
        pred = {(d, f, l) for d, f, l in extracteur(texte)}
        for d, f, l in sorted(vrai - pred):
            print(f"MANQUÉ   {l:11s} {texte[d:f]!r}"); k += 1
        for d, f, l in sorted(pred - vrai):
            print(f"EN TROP  {l:11s} {texte[d:f]!r}"); k += 1
        if k >= n: break

voir_erreurs(test_main, extraire_regles, n=10)

## 3 · La négation et le contexte
Extraire « fièvre » ne suffit pas : dans « **Pas de** fièvre », le patient n'a **pas** de fièvre. Une règle classique (inspirée de *NegEx*) regarde les **mots déclencheurs** juste avant l'entité, dans la même phrase.

👉 **À vous (2/4)** : complétez la liste des déclencheurs de **négation**.

In [ ]:
DECLENCHEURS = {
    "nie":       ["pas de", "sans", "ne présente pas de", "absence de"],
    "incertain": ["suspicion de", "probable", "possible"],
    "famille":   ["antécédent familial de", "antécédents familiaux de"],
}

def statut(texte, debut):
    """Statut d'un problème selon les mots juste avant lui, dans la même phrase."""
    phrase_debut = max(texte.rfind(". ", 0, debut), texte.rfind(" ? ", 0, debut), 0)
    contexte = texte[phrase_debut:debut].lower()
    for nom in ("famille", "incertain", "nie"):
        for dec in DECLENCHEURS[nom]:
            if contexte.rstrip().endswith(dec) or (" " + dec + " ") in (contexte + " "):
                return nom
    return "present"

def extraire_avec_statut(texte):
    return [(d, f, l, statut(texte, d) if l == "PROBLEME" else None) for d, f, l in extraire_regles(texte)]

def score_statut(docs):
    ok = tot = 0
    for texte, ents in docs:
        pred = {(d, f): s for d, f, l, s in extraire_avec_statut(texte) if l == "PROBLEME"}
        for d, f, l, s in ents:
            if l == "PROBLEME" and (d, f) in pred:
                tot += 1; ok += pred[(d, f)] == s
    return round(ok / tot, 2) if tot else None

print("Statut correct (problèmes retrouvés), jeu généré :", score_statut(test_docs))
print("Statut correct (problèmes retrouvés), jeu écrit :", score_statut(test_main))

**Questions :** 
- Quelles formulations de négation un dictionnaire de déclencheurs ne couvrira jamais (« fièvre typhoïde écartée », « n'a jamais eu de… ») ? 
- Pourquoi, dans un dossier médical, vaut-il mieux **signaler** un statut incertain que le deviner ?

In [ ]:
texte = "Pas de toux. Suspicion de diabète. Antécédent familial de diabète (mère). Fièvre depuis 3 jours."
for d, f, l, s in extraire_avec_statut(texte):
    if l == "PROBLEME": print(f"{texte[d:f]!r:12} → {s}")

## 4 · Deuxième approche : un petit étiqueteur appris
Au lieu de listes, on **apprend** à partir d'exemples annotés. On découpe le texte en mots (*tokens*) et on étiquette chaque mot au **format BIO** :
- `B-X` : **début** d'une entité de type X ; `I-X` : **suite** de l'entité ; `O` : hors entité.

Exemple : `goutte` → `B-EXAMEN`, `épaisse` → `I-EXAMEN`, `positive` → `O`.

In [ ]:
def tokeniser(texte):
    return [(m.group(), m.start(), m.end()) for m in re.finditer(r"\w+|[^\w\s]", texte)]

def vers_bio(texte, ents):
    toks = tokeniser(texte)
    bio = ["O"] * len(toks)
    for d, f, l, *_ in ents:
        premier = True
        for i, (w, a, b) in enumerate(toks):
            if a >= d and b <= f:
                bio[i] = ("B-" if premier else "I-") + l; premier = False
    return toks, bio

toks, bio = vers_bio(*test_docs[0])
print(pd.DataFrame({"mot": [t[0] for t in toks], "BIO": bio}).query("BIO != 'O'").head(14).to_string(index=False))

Le modèle regarde, pour chaque mot, **des indices** : le mot lui-même, sa forme (chiffres ? majuscule ?), son début et sa fin, et les mots voisins. Un classeur simple (régression logistique) apprend quelles combinaisons d'indices correspondent à quelle étiquette.

👉 **À vous (3/4)** : ajoutez deux indices de contexte, le **mot précédent** et le **mot suivant**.

In [ ]:
def indices(toks, i):
    w = toks[i][0]
    f = {
        "mot": w.lower(), "suf3": w[-3:].lower(), "pre3": w[:3].lower(),
        "chiffre": w.isdigit(), "majuscule": w[:1].isupper(), "longueur": min(len(w), 12),
    }
    # 👉 Ajoutez le mot précédent et le mot suivant (ou "<debut>" / "<fin>")
    f["prec"] = toks[i - 1][0].lower() if i > 0 else "<debut>"
    f["suiv"] = toks[i + 1][0].lower() if i < len(toks) - 1 else "<fin>"
    return f

def preparer(docs):
    X, y = [], []
    for texte, ents in docs:
        toks, bio = vers_bio(texte, ents)
        for i in range(len(toks)):
            X.append(indices(toks, i)); y.append(bio[i])
    return X, y

Xtr, ytr = preparer(train_docs)
vec = DictVectorizer()
modele = LogisticRegression(max_iter=300)
modele.fit(vec.fit_transform(Xtr), ytr)
print("Entraîné sur", len(ytr), "mots ✔")

In [ ]:
def extraire_ml(texte):
    toks = tokeniser(texte)
    if not toks: return []
    pred = modele.predict(vec.transform([indices(toks, i) for i in range(len(toks))]))
    ents, cour = [], None
    for (w, a, b), p in zip(toks, pred):
        if p.startswith("B-") or (p.startswith("I-") and (cour is None or cour[2] != p[2:])):
            if cour: ents.append(tuple(cour))
            cour = [a, b, p[2:]]
        elif p.startswith("I-"):
            cour[1] = b
        else:
            if cour: ents.append(tuple(cour)); cour = None
    if cour: ents.append(tuple(cour))
    return ents

print("Étiqueteur appris, jeu généré       :", score(test_docs, extraire_ml))
print("Étiqueteur appris, jeu écrit à la main :", score(test_main, extraire_ml))
print("Règles + dictionnaire, jeu écrit à la main :", score(test_main, extraire_regles))

**À observer :** comparez les trois lignes. Avec seulement 300 comptes rendus générés, l'étiqueteur appris n'est pas forcément meilleur que le dictionnaire sur des textes écrits autrement : il manque de **variété** dans ses exemples. Regardez ses erreurs, puis demandez-vous ce qu'il faudrait lui fournir :

In [ ]:
voir_erreurs(test_main, extraire_ml, n=12)

### Piège classique : mesurer par mot ou par entité ?
Un modèle peut étiqueter correctement 95 % des **mots** (la plupart sont `O`) tout en ratant beaucoup d'**entités**. C'est pourquoi l'évaluation se fait **par entité**, avec des bornes exactes, comme ci-dessus.

👉 **À vous (4/4)** : calculez la part de mots correctement étiquetés (la *justesse par mot*) et comparez-la au F1 par entité.

In [ ]:
Xte, yte = preparer(test_docs)
pred = modele.predict(vec.transform(Xte))
par_mot = np.mean(np.array(pred) == np.array(yte))
print("Justesse par mot :", round(float(par_mot), 3))
print("F1 par entité    :", score(test_docs, extraire_ml)["F1"])
print("Part de mots « O » :", round(float(np.mean(np.array(yte) == "O")), 3))

**Point de méthode :** dans un vrai projet, on sépare train et test **par patient** (jamais deux comptes rendus du même patient des deux côtés), sinon les résultats sont gonflés.

## 5 · Passer au scan : l'OCR
Les dossiers papier sont d'abord **photographiés ou scannés**. L'**OCR** (reconnaissance optique de caractères) transforme l'image en texte. Pour le TP, nous **fabriquons** des images de documents fictifs, puis nous les dégradons comme un mauvais scan (légère rotation, flou, bruit, faible contraste).

In [ ]:
def police(taille):
    for chemin in ("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", "DejaVuSans.ttf"):
        try: return ImageFont.truetype(chemin, taille)
        except OSError: pass
    return ImageFont.load_default()

def rendre(lignes, taille=30, degrade=False, seed=0):
    r = random.Random(seed)
    f = police(taille)
    img = Image.new("L", (1100, 60 + 52 * len(lignes)), 255)
    dessin = ImageDraw.Draw(img)
    for i, ligne in enumerate(lignes):
        dessin.text((30, 30 + 52 * i), ligne, font=f, fill=0)
    if degrade:
        img = img.rotate(r.uniform(-2.5, 2.5), expand=True, fillcolor=255)
        img = img.filter(ImageFilter.GaussianBlur(1.1))
        a = np.asarray(img).astype(float)
        a = 255 - (255 - a) * 0.55                       # contraste faible
        a += np.random.RandomState(seed).normal(0, 22, a.shape)  # bruit
        img = Image.fromarray(np.clip(a, 0, 255).astype("uint8"))
    return img

LIGNES = [
    "Patient 021, 34 ans. Consulte le 12/03/2026 pour fievre.",
    "Goutte epaisse positive : paludisme simple.",
    "Traitement : paracetamol 500 mg, 3 fois par jour.",
    "Zinc 20 mg par jour pendant 10 jours.",
    "Tension arterielle 110/70. Revoir dans 2 semaines.",
]
propre = rendre(LIGNES)
sale = rendre(LIGNES, degrade=True, seed=3)
propre.save("scan_propre.png"); sale.save("scan_sale.png")
display(propre); display(sale)

> Les lignes sont écrites **sans accents** pour que la comparaison reste lisible avec n'importe quelle langue installée. Avec `tesseract-ocr-fra`, Tesseract sait lire les accents de documents réels.

### Lire l'image
`pytesseract.image_to_string` renvoie le texte lu.

In [ ]:
texte_propre = pytesseract.image_to_string(propre, lang=LANG)
texte_sale = pytesseract.image_to_string(sale, lang=LANG)
print("--- Scan propre ---"); print(texte_propre)
print("--- Scan dégradé ---"); print(texte_sale)

### Mesurer l'erreur : le CER
Le **CER** (*character error rate*) compte le nombre de modifications (ajouts, suppressions, remplacements de caractères) nécessaires pour retrouver le texte exact, divisé par la longueur du texte exact. **0 = parfait** ; plus c'est haut, plus l'OCR s'est trompé.

👉 **À vous (1/2)** : complétez le calcul du CER avec la distance de Levenshtein fournie.

In [ ]:
def levenshtein(a, b):
    prec = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cour = [i]
        for j, cb in enumerate(b, 1):
            cour.append(min(prec[j] + 1, cour[j - 1] + 1, prec[j - 1] + (ca != cb)))
        prec = cour
    return prec[-1]

def cer(lu, reference):
    lu = " ".join(lu.split()); reference = " ".join(reference.split())
    return levenshtein(lu, reference) / len(reference)

REF = "\n".join(LIGNES)
print("CER scan propre  :", round(cer(texte_propre, REF), 3))
print("CER scan dégradé :", round(cer(texte_sale, REF), 3))

### Améliorer l'image avant la lecture
Redresser, agrandir, augmenter le contraste et **binariser** (noir et blanc) aident souvent beaucoup.

👉**À vous (2/2)** : complétez le seuil de binarisation (un pixel plus clair que le seuil devient blanc, sinon noir). Essayez plusieurs valeurs entre 100 et 200.

In [ ]:
def pretraiter(img, seuil=150, facteur=2):
    img = ImageOps.autocontrast(img)
    img = img.resize((img.width * facteur, img.height * facteur), Image.LANCZOS)
    img = img.filter(ImageFilter.MedianFilter(3))
    return img.point(lambda p: 255 if p > seuil else 0)

amelioree = pretraiter(sale, seuil=150)
display(amelioree)
texte_ameliore = pytesseract.image_to_string(amelioree, lang=LANG)
print(texte_ameliore)
print("CER dégradé brut     :", round(cer(texte_sale, REF), 3))
print("CER après traitement :", round(cer(texte_ameliore, REF), 3))

**À observer :** le traitement améliore-t-il toujours le résultat ? Comparez plusieurs seuils et facteurs. Un traitement trop fort peut **détruire** des traits fins (points, virgules, accents).

In [ ]:
resultats = []
for seuil in (100, 130, 150, 180, 200):
    t = pytesseract.image_to_string(pretraiter(sale, seuil=seuil), lang=LANG)
    resultats.append((seuil, round(cer(t, REF), 3)))
pd.DataFrame(resultats, columns=["seuil", "CER"]).set_index("seuil").T

## 6 · Des erreurs d'OCR qui se propagent
Que devient l'extraction quand le texte vient d'un OCR imparfait ? Comparons les entités extraites du **texte exact** et du **texte lu**. Pour comparer sans dépendre des positions, on compare des paires *(texte de l'entité, étiquette)*.

In [ ]:
def paires(texte):
    return {(texte[d:f].lower(), l) for d, f, l in extraire_ml(texte)}

def comparer(lu, reference):
    a, b = paires(reference), paires(lu)
    vp = len(a & b)
    p = vp / len(b) if b else 0
    r = vp / len(a) if a else 0
    return {"entités attendues": len(a), "retrouvées": vp, "en trop": len(b - a), "rappel": round(r, 2), "précision": round(p, 2)}

print("Texte exact   :", paires(REF) and len(paires(REF)), "entités")
print("Scan dégradé brut      :", comparer(texte_sale, REF))
print("Scan dégradé amélioré  :", comparer(texte_ameliore, REF))
for e in sorted(paires(REF) - paires(texte_sale)): print("  perdue :", e)

**Questions de discussion :**
- Quelles entités sont perdues en premier : les mots courants ou les **chiffres** (doses, dates) ? Pourquoi est-ce le plus dangereux ?
- Que proposez-vous : validation de format (une dose doit ressembler à `nombre + unité`), score de confiance de l'OCR, double saisie des chiffres, relecture humaine des valeurs douteuses ?

### Utiliser la confiance de l'OCR
Tesseract renvoie, pour chaque mot, un **score de confiance**. On peut s'en servir pour **signaler à un humain** les mots douteux plutôt que de les enregistrer sans contrôle.

In [ ]:
d = pytesseract.image_to_data(sale, lang=LANG, output_type=pytesseract.Output.DICT)
mots = pd.DataFrame({"mot": d["text"], "confiance": pd.to_numeric(d["conf"], errors="coerce")})
mots = mots[mots["mot"].str.strip() != ""]
print("Mots à faire relire (confiance < 70) :")
print(mots[mots["confiance"] < 70].to_string(index=False))

## 7 · Pour aller plus loin (facultatif)
Dans un vrai projet, l'étiqueteur appris sur règles est remplacé par un **modèle Transformer** entraîné sur des textes médicaux réels (voir module 3 pour le fine-tuning). La cellule ci-dessous tente de charger un modèle NER **français généraliste** du Hub (`Jean-Baptiste/camembert-ner`, personnes, lieux, organisations) : il **ne connaît pas** les entités cliniques, ce qui illustre le besoin d'un modèle spécialisé. Elle est protégée : le TP continue même si le téléchargement échoue.

In [ ]:
try:
    from transformers import pipeline
    ner = pipeline("token-classification", model="Jean-Baptiste/camembert-ner", aggregation_strategy="simple")
    exemple = "Le docteur Koffi Mensah consulte à l'hôpital de Sokodé le 12 mars."
    for e in ner(exemple): print(f"{e['entity_group']:5s} {e['word']!r:22} score={e['score']:.2f}")
    print()
    for e in ner(test_docs[0][0]): print(f"{e['entity_group']:5s} {e['word']!r:22} score={e['score']:.2f}")
except Exception as err:
    print("Bonus ignoré (modèle non disponible) :", type(err).__name__)

## 8 · Bilan
| Approche | Atouts | Limites |
|---|---|---|
| Règles + dictionnaire | Transparente, rapide à démarrer | Ne généralise pas, liste à maintenir |
| Étiqueteur appris | Généralise un peu aux mots inconnus | Demande des exemples annotés |
| Transformer spécialisé | Meilleur sur textes variés | Données, calcul, gouvernance |
| OCR | Rend les scans exploitables | Erreurs sur les chiffres, qualité d'image |

**À retenir**
1. Évaluer **par entité**, sur un test **séparé par patient**, avec des cas réellement nouveaux.
2. La **négation**, l'**incertitude** et l'**antécédent familial** changent le sens : à traiter explicitement.
3. L'OCR fait des erreurs, surtout sur les **chiffres** : valider les formats et faire **relire** par un humain.
4. Données de santé : autorisation, anonymisation, hébergement sécurisé, traçabilité.

*Pour revenir sur les idées sans code : consultez l'atelier de la version non IT.*